# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/i-am-gk/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Method choice

I will use **Logistic Regression** to predict whether a content item is declining (`down`) from information available before the prediction.

Logistic Regression fits this lane because it is a simple, interpretable classification method and provides a clear ML model to compare against the Week-4 rule-based baseline. I will prioritize interpretability rather than complexity and use the model coefficients to understand which features contribute to the prediction.


In [3]:
# ML-08 Section 1 — Load data and define the modeling target

import pandas as pd
import numpy as np
from pathlib import Path

# Locate the starter dataset in Colab.
csv_path = Path("/content/content_refresh_anonymized.csv")

if not csv_path.exists():
    matches = list(Path("/content").rglob("content_refresh_anonymized.csv"))
    if matches:
        csv_path = matches[0]
    else:
        raise FileNotFoundError(
            "Upload content_refresh_anonymized.csv to the Colab Files panel first."
        )

df = pd.read_csv(csv_path)

# Required fields for this modeling task.
required_cols = [
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
]

missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Define the classification target.
# 1 = declining, 0 = not declining.
df["is_down"] = df["trend_direction"].eq("down").astype(int)

print("Dataset shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())
print("Target distribution:")
print(df["is_down"].value_counts().sort_index())

print("\nTarget rate:")
print(f"Declining: {df['is_down'].mean() * 100:.1f}%")

print("\nSection 1 check: target created successfully.")

Dataset shape: (30000, 45)
Unique clients: 32
Target distribution:
is_down
0    13738
1    16262
Name: count, dtype: int64

Target rate:
Declining: 54.2%

Section 1 check: target created successfully.


### Split design

I will use a **client-grouped train/test split**. All rows belonging to a client will stay in the same split, so the model is not evaluated on content from clients it has already seen during training.

The split is intended to provide a more honest check of whether the model generalizes across clients. The test set will be held out until model evaluation.


In [4]:
# ML-08 Section 2 — Client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

# Features available before prediction.
# Do not include IDs or label-derived fields.
feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
]

# Keep only rows with a complete set of modeling features and a target.
model_df = df[["content_id", "client_id", "is_down"] + feature_cols].dropna().copy()

X = model_df[feature_cols]
y = model_df["is_down"]
groups = model_df["client_id"]

# Split by client so the same client cannot appear in both train and test.
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Rows available for modeling:", len(model_df))
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients.intersection(test_clients)))

print("\nTraining target rate:", f"{y_train.mean() * 100:.1f}%")
print("Test target rate:", f"{y_test.mean() * 100:.1f}%")

print("\nSection 2 check:")
print("Grouped split has no client overlap:", len(train_clients.intersection(test_clients)) == 0)

Rows available for modeling: 20018
Training rows: 14242
Test rows: 5776

Training clients: 23
Test clients: 6
Client overlap: 0

Training target rate: 59.2%
Test target rate: 62.1%

Section 2 check:
Grouped split has no client overlap: True


### Train and compare

I will train Logistic Regression using only pre-decision features and evaluate it on the held-out client groups.

The model will be compared with the Week-4 baseline on the **same test rows and the same metric**. I will use accuracy as a simple classification metric and will report the measured result without treating the model as better merely because it is more complex.


In [5]:
# ML-08 Section 3 — Train Logistic Regression and compare with ML-07 baseline

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ---------------------------------------------------------
# 1. Prepare preprocessing
# ---------------------------------------------------------

# Numeric features are median-imputed and standardized.
# Median imputation is fitted only on the training data through the pipeline.
numeric_features = feature_cols

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        )
    ]
)

# Simple, interpretable classification model.
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    )),
])

# ---------------------------------------------------------
# 2. Train the model
# ---------------------------------------------------------

model.fit(X_train, y_train)

# Model predictions on the held-out test set.
model_pred = model.predict(X_test)

# ---------------------------------------------------------
# 3. Recreate the Week-4 baseline on EXACTLY the same rows
# ---------------------------------------------------------

baseline_test = model_df.iloc[test_idx].copy()

STALE_DAYS = 180
VISIBLE_IMPRESSIONS = 500

baseline_pred = (
    (
        (baseline_test["days_since_last_update"] >= STALE_DAYS) &
        (baseline_test["impressions_90d"] >= VISIBLE_IMPRESSIONS)
    )
    .astype(int)
    .to_numpy()
)

# ---------------------------------------------------------
# 4. Evaluate both using the same test rows and metric
# ---------------------------------------------------------

results = pd.DataFrame([
    {
        "method": "ML-07 rule baseline",
        "accuracy": accuracy_score(y_test, baseline_pred),
        "precision": precision_score(y_test, baseline_pred, zero_division=0),
        "recall": recall_score(y_test, baseline_pred, zero_division=0),
        "f1": f1_score(y_test, baseline_pred, zero_division=0),
    },
    {
        "method": "Logistic Regression",
        "accuracy": accuracy_score(y_test, model_pred),
        "precision": precision_score(y_test, model_pred, zero_division=0),
        "recall": recall_score(y_test, model_pred, zero_division=0),
        "f1": f1_score(y_test, model_pred, zero_division=0),
    },
])

print("=== MODEL VS BASELINE ===")
display(results.round(3))

# ---------------------------------------------------------
# 5. Basic comparison
# ---------------------------------------------------------

baseline_accuracy = results.loc[
    results["method"] == "ML-07 rule baseline", "accuracy"
].iloc[0]

model_accuracy = results.loc[
    results["method"] == "Logistic Regression", "accuracy"
].iloc[0]

print("\nAccuracy difference (model - baseline):",
      round(model_accuracy - baseline_accuracy, 3))

if model_accuracy > baseline_accuracy:
    print("Observed result: Logistic Regression has higher accuracy on this test split.")
elif model_accuracy < baseline_accuracy:
    print("Observed result: the ML-07 baseline has higher accuracy on this test split.")
else:
    print("Observed result: both methods have the same accuracy on this test split.")

print("\nSection 3 check: model and baseline evaluated on the same held-out rows.")

=== MODEL VS BASELINE ===


,method,accuracy,precision,recall,f1
0,ML-07 rule baseline,0.382,0.933,0.004,0.008
1,Logistic Regression,0.766,0.785,0.857,0.820



Accuracy difference (model - baseline): 0.384
Observed result: Logistic Regression has higher accuracy on this test split.

Section 3 check: model and baseline evaluated on the same held-out rows.


### Errors and interpretation

I will inspect the model's incorrect predictions and its strongest coefficients. The goal is to identify where the model makes mistakes and which available features it relies on.

The interpretation will remain directional: the results show patterns observed in this starter snapshot and do not establish that the model will perform the same way on future or production data.


In [6]:
# ML-08 Section 4 — Error analysis and model interpretation

print("=== ERROR ANALYSIS ===")

# Identify incorrect Logistic Regression predictions.
error_mask = model_pred != y_test.to_numpy()

errors = model_df.iloc[test_idx][
    ["content_id", "client_id", "is_down"] + feature_cols
].copy()

errors["prediction"] = model_pred
errors["correct"] = ~error_mask

errors_only = errors[~errors["correct"]].copy()

print("Test rows:", len(y_test))
print("Incorrect predictions:", len(errors_only))
print(
    "Error rate:",
    f"{len(errors_only) / len(y_test) * 100:.1f}%"
)

print("\nPrediction error types:")

false_positives = (
    (model_pred == 1) & (y_test.to_numpy() == 0)
).sum()

false_negatives = (
    (model_pred == 0) & (y_test.to_numpy() == 1)
).sum()

print("False positives:", int(false_positives))
print("False negatives:", int(false_negatives))


# ---------------------------------------------------------
# Inspect a small sample of errors
# ---------------------------------------------------------

print("\n=== SAMPLE ERRORS ===")

error_display_cols = [
    "content_id",
    "is_down",
    "prediction",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "word_count",
]

display(errors_only[error_display_cols].head(10))


# ---------------------------------------------------------
# Model interpretation
# ---------------------------------------------------------

print("\n=== MODEL INTERPRETATION ===")

# Extract the fitted Logistic Regression coefficients.
classifier = model.named_steps["classifier"]
coefficients = classifier.coef_[0]

coef_table = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": coefficients
})

coef_table["abs_coefficient"] = coef_table["coefficient"].abs()

coef_table = coef_table.sort_values(
    "abs_coefficient",
    ascending=False
)

print("Features with the largest absolute coefficients:")

display(
    coef_table[
        ["feature", "coefficient"]
    ].head(10).round(3)
)


# ---------------------------------------------------------
# Plain-language summary
# ---------------------------------------------------------

print("\n=== INTERPRETATION SUMMARY ===")

print(
    f"Logistic Regression made {len(errors_only)} incorrect predictions "
    f"out of {len(y_test)} held-out rows."
)

print(
    f"False positives: {int(false_positives)}; "
    f"false negatives: {int(false_negatives)}."
)

top_feature = coef_table.iloc[0]["feature"]
top_coefficient = coef_table.iloc[0]["coefficient"]

direction = "positive" if top_coefficient > 0 else "negative"

print(
    f"The largest absolute model coefficient was for "
    f"'{top_feature}', with a {direction} coefficient."
)

print(
    "These coefficients describe associations used by the fitted model; "
    "they do not establish causation."
)

print(
    "\nSection 4 check: error counts and model interpretation completed."
)

=== ERROR ANALYSIS ===
Test rows: 5776
Incorrect predictions: 1352
Error rate: 23.4%

Prediction error types:
False positives: 841
False negatives: 511

=== SAMPLE ERRORS ===


,content_id,is_down,prediction,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,word_count
25,content_033ae3e7aecf,1,0,20,27,0,2,2777.0
26,content_72c5c2d73e5a,0,1,13,2426,3,9,2686.0
36,content_bce275871a25,0,1,20,371,5,5,2510.0
43,content_1938955b34c4,1,0,20,184,0,2,2394.0
47,content_40cb4af260c0,1,0,20,8,0,3,3086.0
48,content_326fa2fa449f,1,0,1,4,0,2,3563.0
51,content_d8a23b5e10c5,1,0,8,2,0,3,2756.0
60,content_b9104a222d01,1,0,20,25,1,1,2492.0
64,content_685de0e3b7cb,0,1,8,2639,3,6,2808.0
76,content_5607fec5d7db,1,0,20,22,0,1,2882.0



=== MODEL INTERPRETATION ===
Features with the largest absolute coefficients:


,feature,coefficient
10,impressions_last_30d,-27.771
13,impressions_prev_30d,21.328
11,clicks_last_30d,-1.119
1,impressions_90d,1.051
14,clicks_prev_30d,1.006
4,users_90d,-0.571
8,days_with_impressions,0.561
9,days_with_sessions,-0.549
2,clicks_90d,0.498
3,sessions_90d,0.326



=== INTERPRETATION SUMMARY ===
Logistic Regression made 1352 incorrect predictions out of 5776 held-out rows.
False positives: 841; false negatives: 511.
The largest absolute model coefficient was for 'impressions_last_30d', with a negative coefficient.
These coefficients describe associations used by the fitted model; they do not establish causation.

Section 4 check: error counts and model interpretation completed.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.